## EDA: Factibilidad y objetivos

Se justifica si cada técnica es factible con los datos y qué se espera hallar con cada una. Las cifras vienen de
`a_entendimiento` y `b_exploracion`

- **Corre en:** no ejecuta código (solo texto)

---
### 6. Factibilidad

#### a. Clustering y detección de anomalías (Parte II)

- **86,017 proveedores y 2,996 entidades** con features agregables: tamaño, alcance, dependencia (HHI), competencia, tipo y
  éxito (17 y 14 features)
- Montos con cola pesada (mediana S/192k vs. p99 S/23.1M) → `log1p` antes de estandarizar
- **Mediana de 1 adjudicación por proveedor:** la mayoría tiene un vector casi vacío → filtro de actividad mínima para
  clusterizar proveedores
- KMeans++ escala sobre todos los proveedores (Spark MLlib); DBSCAN/HDBSCAN y jerárquico son O(n²): sobre **todas las
  entidades** (viable en local) y una **muestra estratificada** de proveedores
- **Señales ya visibles:** 36.8% de entidades con HHI > 0.25; métodos con 82-98% de postor único
- **Veredicto:** factible

#### b. Sistemas de recomendación (Parte III)

- Matriz **entidad × proveedor: 2,992 × 86,017, densidad 0.053%**; entidad × familia: 2,992 × 386, densidad 3.4%
- Señal implícita: n.º de adjudicaciones o log1p(monto). No hay ratings → ALS implícito
- Contenido: `descripcion` / `items_desc` por proveedor → TF-IDF (y ANN del P1)
- 36 meses continuos → **hold-out temporal** (2023-2024 → 2025), con `anio` en cada celda para evitar fuga
- **Salvedad:** el 84.9% de proveedores tiene una sola entidad (*cold start*). Se evalúa en entidades con historia
  suficiente y se apoya en la matriz de familias
- **Veredicto:** factible

#### c. Análisis de enlaces: PageRank y HITS (Parte IV)

- **Bipartito dirigido entidad → proveedor: 89,009 nodos y 136,099 aristas**
- Grado con cola pesada (CCDF casi recta en log-log): la centralidad discrimina entre actores
- HITS encaja con la estructura: hubs = entidades, authorities = proveedores
- **Veredicto:** factible. PageRank con GraphFrames sobre el grafo completo; HITS por iteración de potencias sobre la
  matriz dispersa (136k aristas caben en local)

#### d. Grafos y detección de comunidades (Parte V)

- **Co-participación: 60,213 proveedores y 1.64M aristas** (tope 30 participantes, ≥ 2 procesos juntos)
- Pares que compiten juntos más de 2,500 veces: hay estructura de mercado clara
- **Hubs extremos** (grado p99 = 12,059 vs. mediana 16): hay que filtrarlos o normalizarlos, o dominarán las comunidades
- Girvan-Newman y espectral exacto solo en un **subgrafo** (una región o familia, ≤ ~2k nodos), evaluados con modularidad Q
- 10.9% de proveedores en varias regiones y 9.9% en varios segmentos → **BigCLAM** para el solapamiento
- **Veredicto:** factible, con submuestra para los métodos exactos

#### e. Aprendizaje de representación en grafos (Parte VI)

- **SVD truncada** de la matriz entidad × proveedor (136k celdas): embedding directo de ambos tipos de nodo
- **Node2Vec** sobre co-participación (1.6M aristas), sin hubs o en un subgrafo
- Tareas: visualización t-SNE/UMAP contra clústeres y comunidades, y **predicción de enlaces** con split temporal
  (`anio_min` en las aristas)
- **Veredicto:** factible

---
### 7. Objetivos

#### a. Clustering y detección de anomalías

- **Qué se busca:** tipologías de entidades y de proveedores (grandes generalistas, locales de un solo cliente, obras con
  postor único…) y los actores que no encajan en ninguna
- **Anomalías:** actores lejos de su centroide o marcados como ruido por DBSCAN/HDBSCAN, por ejemplo mucho monto con
  poca competencia y un solo cliente
- **Preguntas:** PQ1 (segmento cautivo: HHI alto + postor único + contratación directa) y PQ2 (perfiles de mercado)
- **Artefacto:** `features_proveedor`, `features_entidad`

#### b. Sistemas de recomendación

- **Qué se busca:** para cada entidad, proveedores **candidatos** que entidades parecidas contratan y ella no. Entre ellos,
  **sustitutos** de sus proveedores actuales (mismo rubro y vector parecido)
- **Relaciones atípicas:** relaciones existentes con puntaje muy bajo, que el patrón general no explica
- **Uso descriptivo:** ¿la dependencia de una entidad cautiva se explica por falta de alternativas? No es un ranking de
  calidad ni una sugerencia de a quién adjudicar
- **Preguntas:** PQ3, y PQ1 como complemento
- **Artefacto:** `matriz_entidad_proveedor`, `matriz_entidad_familia`, texto de `procesos`

#### c. Análisis de enlaces: PageRank y HITS

- **Qué se busca:** los proveedores **centrales** del sistema (reciben de muchas entidades importantes) y las entidades
  **hub** (compran a proveedores centrales)
- **Contraste:** centralidad vs. tamaño (monto). Un proveedor muy central con pocas adjudicaciones, o al revés, es
  llamativo
- **Pregunta:** PQ1
- **Artefacto:** `grafo_bipartito`

#### d. Grafos y detección de comunidades

- **Qué se busca:** **mercados** de proveedores que compiten entre sí, y si se organizan por región o por rubro
- **Solapamiento:** proveedores que viven en varios mercados (BigCLAM)
- **Grupos cerrados:** comunidades pequeñas y densas que compiten siempre juntas. Es una señal descriptiva para revisión,
  no una acusación de colusión
- **Pregunta:** PQ2
- **Artefacto:** `grafo_coparticipacion`

#### e. Aprendizaje de representación en grafos

- **Qué se busca:** embeddings de entidades y proveedores (SVD de la matriz, Node2Vec del grafo)
- **Uso:** visualizarlos (t-SNE/UMAP) contra los clústeres de la Parte II y las comunidades de la Parte V (¿convergen?),
  y predecir enlaces nuevos de 2025 entrenando con 2023-2024
- **Preguntas:** PQ2, PQ3
- **Artefacto:** `matriz_entidad_proveedor`, `grafo_coparticipacion`

---
### Resumen

| Técnica | Factible | Pregunta | Artefacto | Salvedad |
|---|---|---|---|---|
| Clustering y anomalías | Sí | PQ1, PQ2 | `features_proveedor`, `features_entidad` | Filtro de actividad; O(n²) en muestra |
| Recomendación | Sí | PQ3 | `matriz_entidad_proveedor`, `matriz_entidad_familia` | *Cold start* (84.9% con 1 entidad) |
| PageRank / HITS | Sí | PQ1 | `grafo_bipartito` | — |
| Comunidades | Sí, con submuestra | PQ2 | `grafo_coparticipacion` | Hubs extremos; métodos exactos en subgrafo |
| Representación | Sí | PQ2, PQ3 | matriz + `grafo_coparticipacion` | Node2Vec sin hubs |